## 🚀 모델 빠른 시작 가이드 (LoRA Inference)

본 저장소에는 모델의 전체 가중치가 아닌, 학습을 통해 변화된 핵심 가중치(**LoRA Adapter**)만 경량화되어 업로드되어 있습니다. 따라서 추론 시 원본 베이스 모델을 먼저 불러온 후 그 위에 파인튜닝된 가중치를 덧씌우는(Merge) 방식으로 작동합니다.

* **환경:** Google Colab (무료 T4 GPU 권장)
* **특징:** 베이스 모델(`Qwen2.5-1.5B-Instruct`)과 `peft` 라이브러리를 활용하여, 적은 VRAM 환경에서도 빠르고 효율적으로 3줄 요약을 수행합니다.

아래 Colab 코드를 순서대로 실행하시면 바로 요약 결과를 확인하실 수 있습니다. 다른 뉴스 기사를 테스트하시려면 코드 내 `article` 변수의 텍스트만 변경해 주시면 됩니다!

In [12]:
# Colab 셀 1: 필수 라이브러리 설치
!pip install -q transformers accelerate torch peft

In [13]:
!pip install -U bitsandbytes>=0.46.1

In [14]:
!pip install -q torchao

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 1. 모델 ID 설정
base_model_id = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
lora_model_id = "kdhhhh1131/ko-news-summarization-model"

print("1. 베이스 모델과 토크나이저를 로드하는 중입니다...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    trust_remote_code=True
)

print("2. LoRA 어댑터를 덧씌우는 중입니다...")
model = PeftModel.from_pretrained(base_model, lora_model_id)

print("모델 로드 완료!\n")

# 3. 테스트할 뉴스 기사 본문
article = """
(도쿄=연합뉴스) 이도연 특파원 = 중국의 대(對)일본 수출 규제 영향으로 지난달 중국의 희토류 자석 대일 수출량이 지난해 5월 이후 1년 만에 최저 수준으로 집계됐다고 니혼게이자이신문(닛케이)이 21일 보도했다.
전날 중국 세관총서가 발표한 무역 통계에 따르면 지난달 중국의 대일 희토류 자석 수출량은 123t으로 전월보다 34.5% 감소했다.
이는 지난해 미중 무역 갈등으로 중국이 희토류 등 수출 통제 강화 조치를 시행했던 지난해 5월 이후 최저 수준으로, 중국의 이중용도(군사·민간 양용) 물자 수출 규제의 영향이 이어지는 것으로 보인다고 닛케이는 전했다.
다카이치 사나에 일본 총리의 '대만 유사시 개입' 시사 발언 이후 중국은 지난 1월 일본에 군사 목적의 이중용도 물자 수출을 금지한 바 있다. 희토류는 이중용도 물자 수출 통제 대상 품목이다.
이런 가운데 반도체 제조 장비를 생산하는 일본 기업들의 지난해 대중국 판매도 감소한 것으로 나타났다.
도쿄 일렉트론, 어드반테스트, 스크린 홀딩스, 디스코, 고쿠사이 일렉트릭 등 일본계 대형 반도체 장비 업체의 2025년도(2025년 4월∼2026년 3월) 중국 매출액은 1조4천700억엔(약 14조원)으로, 전년보다 12% 감소했다.
이들 업체의 중국 매출액이 전년 대비 감소한 것은 이번이 처음이라고 닛케이는 전했다.
이는 중국 정부 주도의 진흥 정책을 통한 장비 국산화가 진행되고 있기 때문이라고 이 신문은 분석했다.
중국의 일본행 관광 규제도 이어지고 있다.
요미우리신문에 따르면 그간 중국 정부의 일본행 단체관광 사실상 금지로 관련 상품 판매를 보류해왔던 중국의 한 국유 여행사가 단체관광 상품 판매를 재개하기로 했다가 돌연 모집을 중단한 것으로 전해졌다.
중국의 대형 국유 여행사 '중국여유그룹'(CTG) 산하 여행사는 오는 7∼8월 6박 7일 일정으로 도쿄와 오사카 등을 방문하는 여행상품 참가자를 모집하고 있었다.
그러나 전날부터 이 업체 홈페이지에는 이 상품이 '판매 중지'로 돼 있어 취소됐을 가능성이 있다고 요미우리신문은 전했다.
관련 보도가 나오면서 중국 정부가 압력을 가했을 수 있다고 이 신문은 덧붙였다.
"""

# 4. Chat Template을 활용한 프롬프트 구성
messages = [
    {"role": "system", "content": "당신은 뉴스 요약 전문가입니다. 반드시 한국어로 대답하세요."},
    {"role": "user", "content": f"다음 뉴스 기사를 읽고 중요한 핵심 팩트만 포함하여 3줄의 불렛 포인트(-)로 요약해 주세요.\n\n[뉴스 기사]\n{article}"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

def get_summary(target_model):
    with torch.no_grad():
        outputs = target_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=200,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("요약을 생성하는 중입니다. 잠시만 기다려주세요...\n")
with model.disable_adapter():
    base_summary = get_summary(model)
lora_summary = get_summary(model)

print("================ [결과 비교] ================\n")
print("☕ [1. 순수 베이스 모델의 요약]")
print(base_summary)
print("-" * 45)
print("✨ [2. 파인튜닝(LoRA) 모델의 요약]")
print(lora_summary)
print("\n=============================================")

1. 베이스 모델과 토크나이저를 로드하는 중입니다...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2. LoRA 어댑터를 덧씌우는 중입니다...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


모델 로드 완료!

요약을 생성하는 중입니다. 잠시만 기다려주세요...



/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


================ [결과 비교] ================

☕ [1. 순수 베이스 모델의 요약]
- 중국 희토류 자석 수출량 현황: 지난달 최저점
- 미국과의 무역纠纷 후 중국의 경쟁력 강화
- 일본 반도체 및 장비업체 중국 매출감소
- 중국 내旅游業 규제 확장 가능성
- 중국 외래주식 소비 금지 여부 의문성
- 일본 여행업자의 판매 결정 변동성
---------------------------------------------
✨ [2. 파인튜닝(LoRA) 모델의 요약]
- 중국의 희토류 자석 수출량이 저감된 후 반도체 장비 및 기타 제품의 중국 매출액 또한 감소했습니다.
- 일본과의 경쟁력 위기 속에서 일본 내외 반도체 업계의 태스크포스가 출범할 것으로 예측됩니다.
- 중국 정부가 일본의 반도체 사업에 대한 지속적인 압력을 가합니다.

